**1. Install and Import Libraries**

In [1]:
!pip install opencv-python mediapipe scikit-learn matplotlib tensorflow


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cv2
import numpy as np
import os
import time
from matplotlib import pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print(os.listdir())

['.git', '.ipynb_checkpoints', '0.npy', 'action_model.keras', 'asl_action_model.h5', 'asl_action_model.keras', 'ASL_Data', 'CodeFile.ipynb', 'hand_landmarker.task', 'Logs', 'pose_landmarker.task', 'Untitled3.ipynb']


**2.Initialize MediaPipe Hand and Pose Landmark Detectors**

In [8]:
# Hand detector
hand_base = python.BaseOptions(model_asset_path="hand_landmarker.task")

hand_options = vision.HandLandmarkerOptions(
    base_options=hand_base,
    num_hands=2
)

hand_detector = vision.HandLandmarker.create_from_options(hand_options)


# Pose detector
pose_base = python.BaseOptions(model_asset_path="pose_landmarker.task")

pose_options = vision.PoseLandmarkerOptions(
    base_options=pose_base
)

pose_detector = vision.PoseLandmarker.create_from_options(pose_options)

In [9]:
def mediapipe_detection(frame):

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    hand_results = hand_detector.detect(mp_image)
    pose_results = pose_detector.detect(mp_image)

    return hand_results, pose_results

In [10]:
def draw_styled_landmarks(frame, hand_results, pose_results):

    # HAND CONNECTIONS
    HAND_CONNECTIONS = [
        (0,1),(1,2),(2,3),(3,4),
        (0,5),(5,6),(6,7),(7,8),
        (5,9),(9,10),(10,11),(11,12),
        (9,13),(13,14),(14,15),(15,16),
        (13,17),(17,18),(18,19),(19,20),
        (0,17)
    ]

    if hand_results.hand_landmarks:
        for hand in hand_results.hand_landmarks:

            # draw points
            for lm in hand:
                x = int(lm.x * frame.shape[1])
                y = int(lm.y * frame.shape[0])
                cv2.circle(frame, (x,y), 4, (0,255,0), -1)

            # draw lines
            for connection in HAND_CONNECTIONS:
                start = hand[connection[0]]
                end = hand[connection[1]]

                x1 = int(start.x * frame.shape[1])
                y1 = int(start.y * frame.shape[0])
                x2 = int(end.x * frame.shape[1])
                y2 = int(end.y * frame.shape[0])

                cv2.line(frame, (x1,y1), (x2,y2), (0,255,255), 2)


    # POSE CONNECTIONS
    POSE_CONNECTIONS = [
        (11,13),(13,15),
        (12,14),(14,16),
        (11,12)
    ]

    if pose_results.pose_landmarks:

        pose = pose_results.pose_landmarks[0]

        for lm in pose:
            x = int(lm.x * frame.shape[1])
            y = int(lm.y * frame.shape[0])
            cv2.circle(frame, (x,y), 3, (255,0,0), -1)

        for connection in POSE_CONNECTIONS:
            start = pose[connection[0]]
            end = pose[connection[1]]

            x1 = int(start.x * frame.shape[1])
            y1 = int(start.y * frame.shape[0])
            x2 = int(end.x * frame.shape[1])
            y2 = int(end.y * frame.shape[0])

            cv2.line(frame, (x1,y1), (x2,y2), (255,255,0), 2)

In [11]:
cap = cv2.VideoCapture(0)

In [12]:
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    hand_results, pose_results = mediapipe_detection(frame)

    draw_styled_landmarks(frame, hand_results, pose_results)

    cv2.imshow("ASL Detection Feed", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

**3.Extract Landmark Feature Vectors for Model Training**

In [13]:
def extract_pose(pose_results):

    if pose_results.pose_landmarks:

        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility]
             for lm in pose_results.pose_landmarks[0]]
        ).flatten()

    else:
        pose = np.zeros(33*4)

    return pose

In [14]:
def extract_hands(hand_results):

    left = np.zeros(21*3)
    right = np.zeros(21*3)

    if hand_results.hand_landmarks:

        for idx, hand in enumerate(hand_results.hand_landmarks):

            hand_array = np.array(
                [[lm.x, lm.y, lm.z] for lm in hand]
            ).flatten()

            if idx == 0:
                left = hand_array
            elif idx == 1:
                right = hand_array

    return left, right

In [15]:
def extract_keypoints(hand_results, pose_results):

    pose = extract_pose(pose_results)

    left, right = extract_hands(hand_results)

    return np.concatenate([pose, left, right])

In [16]:
keypoints = extract_keypoints(hand_results, pose_results)

print("Feature vector length:", len(keypoints))

Feature vector length: 258


In [17]:
result_test = extract_keypoints(hand_results, pose_results)

In [18]:
33*4 + 21*3 + 21*3

258

In [19]:
np.save('0', result_test)
np.load('0.npy')

array([ 6.04851365e-01,  6.97021723e-01, -2.03304005e+00,  9.99626517e-01,
        6.42255545e-01,  6.10716343e-01, -1.94649315e+00,  9.99430478e-01,
        6.63543284e-01,  6.10935688e-01, -1.94663393e+00,  9.99321342e-01,
        6.80853486e-01,  6.10750437e-01, -1.94712043e+00,  9.99045551e-01,
        5.69200158e-01,  6.10624075e-01, -1.95763588e+00,  9.99439895e-01,
        5.41307271e-01,  6.11186147e-01, -1.95806646e+00,  9.99359548e-01,
        5.19432664e-01,  6.12627506e-01, -1.95803642e+00,  9.99150991e-01,
        6.98493421e-01,  6.38290048e-01, -1.31464827e+00,  9.98808384e-01,
        4.97352183e-01,  6.40934348e-01, -1.38542914e+00,  9.98952985e-01,
        6.47164226e-01,  7.64651060e-01, -1.77182651e+00,  9.93111730e-01,
        5.60949862e-01,  7.62602568e-01, -1.79159558e+00,  9.95202899e-01,
        8.02137852e-01,  1.01822019e+00, -9.20332134e-01,  5.43467104e-01,
        3.15802127e-01,  9.91965175e-01, -9.71441269e-01,  7.54040301e-01,
        8.01633418e-01,  

**4.Create Dataset Directory Structure for Training Data**

In [20]:
import os

In [21]:
DATA_PATH = os.path.join("ASL_Data")

In [22]:
no_sequences = 30
sequence_length = 30

In [23]:
actions = np.array([
    "hello",
    "thanks",
    "iloveyou",
    "yes",
    "no",
    "please",
    "sorry",
    "help",
    "stop",
    "good",
    "bad",
    "bye"
])

In [24]:
for action in actions:
    for sequence in range(no_sequences):
        dir_path = os.path.join(DATA_PATH, action, str(sequence))
        os.makedirs(dir_path, exist_ok=True)

**5.Collect Training Sequences and Save Landmark Keypoints**

In [22]:
cap = cv2.VideoCapture(0)

In [23]:
stop_collection = False 

In [24]:
for action in actions:
    for sequence in range(no_sequences):
        for frame_num in range(sequence_length):

            ret, frame = cap.read()
            if not ret:
                stop_collection = True
                break

            hand_results, pose_results = mediapipe_detection(frame) # Run MediaPipe detection
            draw_styled_landmarks(frame, hand_results, pose_results) # Draw landmarks
            keypoints = extract_keypoints(hand_results, pose_results) # Extract keypoints

            npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num)) # Save keypoints
            np.save(npy_path, keypoints)

            # Display status on screen
            cv2.putText(
                frame,
                f"Collecting {action} | Video {sequence}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            cv2.imshow("ASL Data Collection", frame)

            if cv2.waitKey(10) & 0xFF == ord('q'):
                stop_collection = True
                break

        if stop_collection:
            break
    if stop_collection:
        break

In [25]:
cap.release()
cv2.destroyAllWindows()

**6.Dataset Assembly & Train/Test Split**

In [25]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [26]:
label_map = {label: num for num, label in enumerate(actions)}
print(label_map)

{np.str_('hello'): 0, np.str_('thanks'): 1, np.str_('iloveyou'): 2, np.str_('yes'): 3, np.str_('no'): 4, np.str_('please'): 5, np.str_('sorry'): 6, np.str_('help'): 7, np.str_('stop'): 8, np.str_('good'): 9, np.str_('bad'): 10, np.str_('bye'): 11}


In [27]:
sequences = []
labels = []

for action in actions:
    for sequence in range(no_sequences):

        window = []
        valid_sequence = True

        for frame_num in range(sequence_length):

            file_path = os.path.join(
                DATA_PATH, action, str(sequence), f"{frame_num}.npy"
            )

            if not os.path.exists(file_path):
                valid_sequence = False
                break

            res = np.load(file_path)
            window.append(res)

        if valid_sequence:
            sequences.append(window)
            labels.append(label_map[action])

In [28]:
X = np.array(sequences)
y = to_categorical(labels, num_classes=len(actions)).astype(int)

print("X shape:",X.shape)
print("y shape:",y.shape)

X shape: (175, 30, 258)
y shape: (175, 12)


In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    stratify=labels,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (140, 30, 258)
Test: (35, 30, 258)


In [30]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(140, 30, 258)
(140, 12)
(35, 30, 258)
(35, 12)


**7.Build and train LSTM Neural Network**

In [31]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.callbacks import TensorBoard
import os

In [32]:
log_dir = os.path.join("Logs")
tb_callback = TensorBoard(log_dir=log_dir)

from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

In [33]:
model = Sequential()

model.add(Input(shape=(30, X.shape[2])))

model.add(LSTM(64, return_sequences=True))
model.add(LSTM(128, return_sequences=True))
model.add(LSTM(64))

model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))

model.add(Dense(actions.shape[0], activation='softmax'))

In [34]:
model.compile(
    optimizer='Adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [35]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 30, 64)              │          82,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 30, 128)             │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_2 (LSTM)                        │ (None, 64)                  │          49,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 12)                  │             396 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 237,548 (927.92 KB)

 Trainable params: 237,548 (927.92 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
pip install tensorboard

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: C:\Users\nithy\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [37]:
model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[tb_callback, early_stop]
)

Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 11s 203ms/step - accuracy: 0.1714 - loss: 2.2607 - val_accuracy: 0.1714 - val_loss: 2.0382
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - accuracy: 0.1071 - loss: 1.9726 - val_accuracy: 0.1714 - val_loss: 1.8794
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.1357 - loss: 1.8807 - val_accuracy: 0.1714 - val_loss: 1.8229
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.1357 - loss: 1.8615 - val_accuracy: 0.1714 - val_loss: 1.8097
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.2000 - loss: 1.8084 - val_accuracy: 0.3143 - val_loss: 1.7855
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.2429 - loss: 1.7680 - val_accuracy: 0.2571 - val_loss: 1.7420
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - accuracy: 0.2571 - loss: 1.7197 - val_accuracy: 0.2286 - val_loss: 1.7327
Epoch 8/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.3214 - loss: 1.6443 - val_accuracy: 0.2571 - val_los

In [38]:
model.save("asl_action_model.keras")

**8.Make Predictions**

In [39]:
res = model.predict(X_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 893ms/step


In [40]:
actions[np.argmax(res[4])]

np.str_('yes')

In [41]:
actions[np.argmax(y_test[4])]

np.str_('yes')

**9.Save Weights**

In [42]:
model.save("asl_action_model.keras")

In [43]:
from tensorflow.keras.models import load_model

In [44]:
model = load_model("asl_action_model.keras")

**10.Evaluation using Confusion Matrix and Accuracy**

In [45]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [46]:
y_pred = model.predict(X_train)

5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 211ms/step


In [47]:
y_true = np.argmax(y_train, axis=1)
y_pred = np.argmax(y_pred, axis=1)

In [48]:
multilabel_confusion_matrix(y_true, y_pred)

array([[[116,   0],
        [  0,  24]],

       [[116,   0],
        [  0,  24]],

       [[116,   0],
        [  0,  24]],

       [[116,   0],
        [  1,  23]],

       [[115,   1],
        [  0,  24]],

       [[120,   0],
        [  0,  20]]])

In [49]:
y_pred = model.predict(X_test)

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(y_pred, axis=1)

multilabel_confusion_matrix(y_true, y_pred)
accuracy_score(y_true, y_pred)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step


0.9714285714285714

**11.Real_Time Testing**

In [50]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

In [51]:
model = load_model("asl_action_model.keras")

In [70]:
cap = cv2.VideoCapture(0)

In [71]:
sequence = []
sentence = []
predictions=[]
threshold = 0.8

In [72]:
while True:

    ret, frame = cap.read()
    if not ret:
        print("Frame not captured")
        break

    # MediaPipe detection
    hand_results, pose_results = mediapipe_detection(frame)

    # Draw landmarks
    draw_styled_landmarks(frame, hand_results, pose_results)

    # Extract keypoints
    keypoints = extract_keypoints(hand_results, pose_results)

    sequence.append(keypoints)
    sequence = sequence[-30:]

    threshold=0.8

    # Prediction
    if len(sequence) == 30:

        res = model.predict(np.expand_dims(sequence, axis=0))[0]

        predictions.append(np.argmax(res))
        predictions = predictions[-10:]

        if np.unique(predictions)[0] == np.argmax(res):

            if res[np.argmax(res)] > threshold:

                if len(sentence) == 0 or actions[np.argmax(res)] != sentence[-1]:
                    sentence.append(actions[np.argmax(res)])

        if len(sentence) > 5:
            sentence = sentence[-5:]

    # Display prediction
    cv2.putText(
        frame,
        ' '.join(sentence),
        (20,450),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (255,255,255),
        2,
        cv2.LINE_AA
    )

    cv2.imshow("ASL Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━

In [73]:
cap.release()
cv2.destroyAllWindows()

In [74]:
res = model.predict(X_test)

for i in range(10):
    print("Predicted:", actions[np.argmax(res[i])],
          " | True:", actions[np.argmax(y_test[i])])

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Predicted: no  | True: no
Predicted: thanks  | True: thanks
Predicted: please  | True: please
Predicted: iloveyou  | True: no
Predicted: yes  | True: yes
Predicted: iloveyou  | True: iloveyou
Predicted: thanks  | True: thanks
Predicted: hello  | True: hello
Predicted: no  | True: no
Predicted: iloveyou  | True: iloveyou
